In [ ]:
import ants
import nibabel as nib
import numpy as np
from nilearn import plotting

In [ ]:
# --- Parameters ---
SUB = 'sub-03'   # adjust
CAP = 'cap2'     # adjust
RUN = 'run2'     # adjust — the problematic one
PROJECT_DIR = '/data/Annie/PROJECTS/EEGoddball_ERP_1'  # adjust

# --- Paths ---
ANAT_DIR    = f'{PROJECT_DIR}/derivatives/{SUB}/preproc/T1'
PREPROC_DIR = f'{PROJECT_DIR}/derivatives/{SUB}/preproc/{CAP}'
OUTPUT      = f'{PROJECT_DIR}/derivatives/{SUB}'

t1_path   = f'{ANAT_DIR}/UNI_T1.nii'
func_path = f'{PREPROC_DIR}/mc/meanFunctional_{RUN}.nii.gz'
# mask_out  = f'{PREPROC_DIR}/FIACH/rfBrainMask_{RUN}.nii'   # only if running after preprocessing with issues
brain_out = f'{OUTPUT}/rfBrainMask_{RUN}_test.nii'

# --- Load ---
img_t1   = ants.image_read(t1_path)
meanFunc = ants.image_read(func_path)

# --- Register ---
registration = ants.registration(
    fixed=meanFunc, moving=img_t1,
    type_of_transform='BOLDRigid',
    random_seed=100,
    verbose=True
)

# --- Apply transform to mask ---
mask_t1_path = f'{ANAT_DIR}/rfBrainMask_T1space.nii'
mask_t1      = ants.image_read(mask_t1_path)

mask_epi = ants.apply_transforms(
    fixed=meanFunc, moving=mask_t1,
    transformlist=registration['fwdtransforms'],
    interpolator='nearestNeighbor'
)

# --- EPI signal clipping ---
meanfunc_nib = nib.load(func_path)
epi_data     = meanfunc_nib.get_fdata()
epi_thresh   = np.percentile(epi_data, 5)
epi_signal   = (epi_data > epi_thresh).astype(np.float32)
mask_final   = (mask_epi.numpy() > 0.5).astype(np.float32)
mask_final   = (mask_final * epi_signal).astype(np.float32)

warped_t1    = registration['warpedmovout'].numpy()
brain_masked = (warped_t1 * mask_final).astype(np.float32)

# --- Save test output (does not overwrite existing mask_out) ---
nib.save(nib.Nifti1Image(mask_final,   meanfunc_nib.affine, meanfunc_nib.header), brain_out)
nib.save(nib.Nifti1Image(brain_masked, meanfunc_nib.affine, meanfunc_nib.header),
         brain_out.replace('rfBrainMask', 'Masked_UNI'))

print(f'Brain voxels: {int(mask_final.sum())}')

# --- QC ---
plotting.plot_roi(
    roi_img=brain_out,
    bg_img=func_path,
    display_mode='mosaic', cut_coords=7, alpha=0.4,
    title=f'TEST rfBrainMask — {SUB}_{CAP}_{RUN}'
)
## only if running after preprocessing with issues
# plotting.plot_roi( 
#     roi_img=mask_out,
#     bg_img=func_path,
#     display_mode='mosaic', cut_coords=7, alpha=0.4,
#     title=f'ORIGINAL rfBrainMask — {SUB}_{CAP}_{RUN}'
# )      
plotting.show()

In [ ]:
# # COM initialisation — replaces initial_transform='com'
# fixed_com  = np.array(ants.get_center_of_mass(meanFunc))
# moving_com = np.array(ants.get_center_of_mass(img_t1))
# translation = fixed_com - moving_com

# init_tx = ants.new_ants_transform(
#     transform_type='Euler3DTransform',
#     precision='float',
#     dimension=3
# )
# params = init_tx.parameters
# params[3:6] = translation          # last 3 params are translation
# init_tx.set_parameters(params)

# init_path = '/tmp/com_init.mat'
# ants.write_transform(init_tx, init_path)

# registration = ants.registration(
#     fixed=meanFunc,
#     moving=img_t1,
#     type_of_transform='Rigid',
#     initial_transform=init_path,    # now a real file path
#     metric='mattes',
#     metric_weight=1,
#     radius_or_number_of_bins=64,
#     sampling_strategy='random',
#     sampling_percentage=0.5,
#     number_of_iterations=[[1000, 500, 250, 100]],
#     convergence_threshold=1e-7,
#     convergence_window_size=10,
#     smoothing_sigmas=[[3, 2, 1, 0]],
#     shrink_factors=[[8, 4, 2, 1]],
#     use_histogram_matching=False,
#     random_seed=42,
#     verbose=True
# )

In [ ]:
# import subprocess, tempfile, os

# warped_t1_path = f'{OUTPUT}/warped_t1_{RUN}.nii.gz'
# synthseg_path  = f'{OUTPUT}/synthseg_epispace_{RUN}.nii.gz'
# SYNTHSEG_BIN   = '/usr/local/freesurfer/8.2.0/bin/mri_synthseg'
# warped_t1_ants = registration['warpedmovout']

# ants.image_write(warped_t1_ants, warped_t1_path)

# result = subprocess.run(
#     'export FREESURFER_HOME=/usr/local/freesurfer/8.2.0 && '
#     'source $FREESURFER_HOME/SetUpFreeSurfer.sh && '
#     f'{SYNTHSEG_BIN} --i {warped_t1_path} --o {synthseg_path} --robust --threads 8',
#     shell=True, executable='/bin/bash',
#     capture_output=True, text=True
# )

# print('RETURN CODE:', result.returncode)
# print('STDOUT:', result.stdout)
# print('STDERR:', result.stderr)

In [ ]:
# from nilearn.image import resample_to_img

# seg_nib    = nib.load(synthseg_path)
# meanfunc_nib = nib.load(func_path)

# # Resample segmentation to EPI space before binarising
# seg_epi    = resample_to_img(seg_nib, meanfunc_nib, interpolation='nearest')
# seg_data   = seg_epi.get_fdata()
# mask_final = (seg_data >= 1).astype(np.float32)

# # --- EPI signal clipping ---
# epi_data   = meanfunc_nib.get_fdata()
# epi_thresh = np.percentile(epi_data, 5)
# epi_signal = (epi_data > epi_thresh).astype(np.float32)
# mask_final = (mask_final * epi_signal).astype(np.float32)

# warped_t1    = warped_t1_ants.numpy()
# brain_masked = (warped_t1 * mask_final).astype(np.float32)

# # --- Save ---
# nib.save(nib.Nifti1Image(mask_final,   meanfunc_nib.affine, meanfunc_nib.header), brain_out)
# nib.save(nib.Nifti1Image(brain_masked, meanfunc_nib.affine, meanfunc_nib.header),
#          brain_out.replace('rfBrainMask', 'Masked_UNI'))

# print(f'Brain voxels: {int(mask_final.sum())}')

# # --- QC plots ---
# plotting.plot_roi(
#     roi_img=brain_out,
#     bg_img=func_path,
#     display_mode='mosaic', cut_coords=7, alpha=0.4,
#     title=f'TEST rfBrainMask — {RUN}'
# )
# plotting.plot_roi(
#     roi_img=mask_out,
#     bg_img=func_path,
#     display_mode='mosaic', cut_coords=7, alpha=0.4,
#     title=f'ORIGINAL rfBrainMask — {RUN}'
# )
# plotting.show()

In [ ]:
# from nilearn.image import resample_to_img

# # Resample warped T1 to EPI grid BEFORE SynthSeg
# warped_t1_nib     = nib.load(warped_t1_path)
# warped_t1_epi_nib = resample_to_img(warped_t1_nib, meanfunc_nib, interpolation='linear')
# warped_t1_epi_path = f'{OUTPUT}/warped_t1_{RUN}_epispace.nii.gz'
# nib.save(warped_t1_epi_nib, warped_t1_epi_path)

# print('warped T1 shape:    ', warped_t1_nib.shape)
# print('warped T1 EPI shape:', warped_t1_epi_nib.shape)
# print('meanFunc shape:     ', meanfunc_nib.shape)

# # Run SynthSeg on the EPI-grid warped T1
# result = subprocess.run(
#     'export FREESURFER_HOME=/usr/local/freesurfer/8.2.0 && '
#     'source $FREESURFER_HOME/SetUpFreeSurfer.sh && '
#     f'{SYNTHSEG_BIN} --i {warped_t1_epi_path} --o {synthseg_path} --robust --threads 8',
#     shell=True, executable='/bin/bash',
#     capture_output=True, text=True
# )

# print('RETURN CODE:', result.returncode)
# print('STDOUT:', result.stdout)

# seg_data   = nib.load(synthseg_path).get_fdata()
# mask_final = (seg_data >= 1).astype(np.float32)

# print('seg shape:     ', seg_data.shape)
# print('meanFunc shape:', meanfunc_nib.get_fdata().shape)

In [ ]:
# # Warped T1 as background for both masks
# plotting.plot_roi(
#     roi_img=brain_out,
#     bg_img=warped_t1_epi_path,
#     display_mode='mosaic', cut_coords=7, alpha=0.4,
#     title=f'TEST mask on warped T1 — {SUB}_{CAP}_{RUN}'
# )
# plotting.plot_roi(
#     roi_img=mask_out,
#     bg_img=warped_t1_epi_path,
#     display_mode='mosaic', cut_coords=7, alpha=0.4,
#     title=f'ORIGINAL mask on warped T1 — {SUB}_{CAP}_{RUN}'
# )

# # meanFunc as background for both masks
# plotting.plot_roi(
#     roi_img=brain_out,
#     bg_img=func_path,
#     display_mode='mosaic', cut_coords=7, alpha=0.4,
#     title=f'TEST mask on meanFunc — {SUB}_{CAP}_{RUN}'
# )
# plotting.plot_roi(
#     roi_img=mask_out,
#     bg_img=func_path,
#     display_mode='mosaic', cut_coords=7, alpha=0.4,
#     title=f'ORIGINAL mask on meanFunc — {SUB}_{CAP}_{RUN}'
# )
# plotting.show()

In [ ]:
# synthseg_epi_path = f'{OUTPUT}/synthseg_directEPI_{RUN}.nii.gz'

# result = subprocess.run(
#     'export FREESURFER_HOME=/usr/local/freesurfer/8.2.0 && '
#     'source $FREESURFER_HOME/SetUpFreeSurfer.sh && '
#     f'{SYNTHSEG_BIN} --i {func_path} --o {synthseg_epi_path} --robust --threads 8',
#     shell=True, executable='/bin/bash',
#     capture_output=True, text=True
# )

# print('RETURN CODE:', result.returncode)
# print('STDOUT:', result.stdout)

# seg_data   = nib.load(synthseg_epi_path).get_fdata()
# mask_final = (seg_data >= 1).astype(np.float32)

# nib.save(nib.Nifti1Image(mask_final, meanfunc_nib.affine, meanfunc_nib.header), brain_out)

# plotting.plot_roi(
#     roi_img=brain_out,
#     bg_img=func_path,
#     display_mode='mosaic', cut_coords=7, alpha=0.4,
#     title=f'Direct EPI SynthSeg mask — {SUB}_{CAP}_{RUN}'
# )
# plotting.plot_roi(
#     roi_img=mask_out,
#     bg_img=func_path,
#     display_mode='mosaic', cut_coords=7, alpha=0.4,
#     title=f'ORIGINAL mask — {SUB}_{CAP}_{RUN}'
# )
# plotting.show()

In [ ]:
# import nibabel as nib

# # fill in paths for a GOOD run and a BAD run
# good_func = mask_out  
# bad_func  = brain_out   
# t1        = t1_path  
# func = func_path

# good_nib = nib.load(good_func)
# bad_nib  = nib.load(bad_func)
# t1_nib   = nib.load(t1)
# func_nib = nib.load(func)

# print('=== T1 ===')
# print('affine:\n', t1_nib.affine)
# print('sform code:', t1_nib.header.get_sform(coded=True)[1])
# print('qform code:', t1_nib.header.get_qform(coded=True)[1])

# print('=== meanFunc ===')
# print('affine:\n', func_nib.affine)
# print('sform code:', func_nib.header.get_sform(coded=True)[1])
# print('qform code:', func_nib.header.get_qform(coded=True)[1])

# print('\n=== GOOD mask ===')
# print('affine:\n', good_nib.affine)
# print('sform code:', good_nib.header.get_sform(coded=True)[1])
# print('qform code:', good_nib.header.get_qform(coded=True)[1])

# print('\n=== BAD mask ===')
# print('affine:\n', bad_nib.affine)
# print('sform code:', bad_nib.header.get_sform(coded=True)[1])
# print('qform code:', bad_nib.header.get_qform(coded=True)[1])

In [ ]:
import ants

print('T1 COM:       ', ants.get_center_of_mass(ants.image_read(t1_path)))
print('meanFunc COM: ', ants.get_center_of_mass(ants.image_read(func_path)))
print('distance (mm):', np.linalg.norm(
    np.array(ants.get_center_of_mass(ants.image_read(t1_path))) -
    np.array(ants.get_center_of_mass(ants.image_read(func_path)))
))